In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from datetime import datetime

# Cargar datos
df = pd.read_csv('data/datos.csv')

# Limpiar datos
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Crear variables adicionales para análisis
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 18, 35, 60, 100], 
                       labels=['Niño/Adolescente', 'Adulto Joven', 'Adulto', 'Adulto Mayor'])
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Calcular métricas básicas
survived_count = df['Survived'].sum()
died_count = len(df) - df['Survived'].sum()
survival_rate = df['Survived'].mean() * 100
total_passengers = len(df)
current_date = datetime.now().strftime("%d/%m/%Y")

print(f"Dataset cargado exitosamente: {total_passengers} registros")

## Resumen Ejecutivo {.sidebar}

::: {.card}
**Profesor:** GINO JOEL TAIPE MIRANDA

**Estudiante:** RUIZ ALVA JERSON ENMANUEL

**Fecha:** 

In [ ]:
print(current_date)

**Dataset:** Titanic - 891 pasajeros
:::

::: {.card}
### Hallazgos Clave
- Tasa de supervivencia general: **38.4%**
- Mayor supervivencia en **primera clase**
- Las **mujeres** tuvieron mayor tasa de supervivencia
- La **edad** influyó en las probabilidades de supervivencia
:::

# Análisis Descriptivo

## Row {height=60%}

In [ ]:
#| title: "Distribución de Supervivencia por Clase y Género"

fig = px.sunburst(df, 
                  path=['Pclass', 'Sex', 'Survived'], 
                  values='PassengerId',
                  color='Survived',
                  color_discrete_map={0: '#ff6b6b', 1: '#51cf66'},
                  title="Supervivencia por Clase Socioeconómica y Género")
fig.update_layout(height=500)
fig.show()

In [ ]:
#| title: "Tasa de Supervivencia por Grupo de Edad"

survival_by_age = df.groupby('AgeGroup')['Survived'].agg(['mean', 'count']).reset_index()
survival_by_age['percentage'] = survival_by_age['mean'] * 100

fig = px.bar(survival_by_age, 
             x='AgeGroup', 
             y='percentage',
             text='count',
             title="Porcentaje de Supervivencia por Grupo Etario",
             labels={'percentage': 'Tasa de Supervivencia (%)', 'AgeGroup': 'Grupo de Edad'},
             color='percentage',
             color_continuous_scale='RdYlGn')
fig.update_traces(texttemplate='n=%{text}', textposition='outside')
fig.update_layout(height=500)
fig.show()

## Row {height=40%}

::: {.valuebox icon="person-check" color="success"}
Supervivientes

In [ ]:
print(f"{survived_count:,}")

:::

::: {.valuebox icon="person-x" color="danger"}
Fallecidos

In [ ]:
print(f"{died_count:,}")

:::

::: {.valuebox icon="percent" color="info"}
Tasa Supervivencia

In [ ]:
print(f"{survival_rate:.1f}%")

:::

::: {.valuebox icon="people" color="primary"}
Total Pasajeros

In [ ]:
print(f"{total_passengers:,}")

:::

# Análisis Diagnóstico

## Row

In [ ]:
#| title: "Factores de Riesgo: Análisis Multivariado"

# Crear subplot con múltiples análisis
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Supervivencia por Clase', 'Supervivencia por Género', 
                   'Distribución de Edad', 'Tamaño de Familia vs Supervivencia'),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "histogram"}, {"type": "scatter"}]]
)

# Gráfico 1: Supervivencia por clase
class_survival = df.groupby('Pclass')['Survived'].mean()
fig.add_trace(go.Bar(x=class_survival.index, y=class_survival.values, 
                     name='Por Clase', marker_color='lightblue'), row=1, col=1)

# Gráfico 2: Supervivencia por género
gender_survival = df.groupby('Sex')['Survived'].mean()
fig.add_trace(go.Bar(x=gender_survival.index, y=gender_survival.values, 
                     name='Por Género', marker_color='lightgreen'), row=1, col=2)

# Gráfico 3: Distribución de edad
fig.add_trace(go.Histogram(x=df['Age'], name='Distribución Edad', 
                          marker_color='lightyellow'), row=2, col=1)

# Gráfico 4: Familia vs supervivencia
family_survival = df.groupby('FamilySize')['Survived'].mean()
fig.add_trace(go.Scatter(x=family_survival.index, y=family_survival.values, 
                        mode='markers+lines', name='Familia vs Supervivencia',
                        marker_color='lightcoral'), row=2, col=2)

fig.update_layout(height=600, showlegend=False)
fig.show()

In [ ]:
#| title: "Matriz de Correlación de Variables Numéricas"

# Seleccionar variables numéricas
numeric_vars = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']
correlation_matrix = df[numeric_vars].corr()

fig = px.imshow(correlation_matrix, 
                text_auto=True, 
                aspect="auto",
                color_continuous_scale='RdBu',
                title="Correlaciones entre Variables")
fig.update_layout(height=500)
fig.show()

# Análisis Predictivo

## Row

In [ ]:
#| title: "Modelo de Supervivencia: Probabilidades por Segmento"

# Crear análisis predictivo simple basado en patrones observados
def predict_survival_probability(row):
    prob = 0.5  # Base probability
    
    # Ajustar por clase
    if row['Pclass'] == 1:
        prob += 0.3
    elif row['Pclass'] == 2:
        prob += 0.1
    else:
        prob -= 0.2
    
    # Ajustar por género
    if row['Sex'] == 'female':
        prob += 0.4
    else:
        prob -= 0.3
    
    # Ajustar por edad
    if row['Age'] < 18:
        prob += 0.1
    elif row['Age'] > 60:
        prob -= 0.1
    
    # Ajustar por tamaño de familia
    if 2 <= row['FamilySize'] <= 4:
        prob += 0.1
    elif row['FamilySize'] > 4:
        prob -= 0.2
    
    return max(0, min(1, prob))

df['PredictedSurvivalProb'] = df.apply(predict_survival_probability, axis=1)

# Crear visualización de predicciones
survival_segments = df.groupby(['Pclass', 'Sex']).agg({
    'PredictedSurvivalProb': 'mean',
    'Survived': 'mean',
    'PassengerId': 'count'
}).reset_index()

fig = px.scatter(survival_segments, 
                x='PredictedSurvivalProb', 
                y='Survived',
                size='PassengerId',
                color='Pclass',
                hover_data=['Sex'],
                title="Probabilidad Predicha vs Supervivencia Real",
                labels={'PredictedSurvivalProb': 'Probabilidad Predicha',
                       'Survived': 'Tasa de Supervivencia Real'})

# Agregar línea de predicción perfecta
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', 
                        name='Predicción Perfecta', line=dict(dash='dash')))
fig.update_layout(height=500)
fig.show()

In [ ]:
#| title: "Escenarios de Supervivencia: Análisis 'What-if'"

# Crear diferentes escenarios
scenarios = [
    {'name': 'Mujer, 1ra Clase, Adulta', 'Pclass': 1, 'Sex': 'female', 'Age': 30, 'FamilySize': 2},
    {'name': 'Hombre, 1ra Clase, Adulto', 'Pclass': 1, 'Sex': 'male', 'Age': 30, 'FamilySize': 2},
    {'name': 'Mujer, 3ra Clase, Adulta', 'Pclass': 3, 'Sex': 'female', 'Age': 30, 'FamilySize': 2},
    {'name': 'Hombre, 3ra Clase, Adulto', 'Pclass': 3, 'Sex': 'male', 'Age': 30, 'FamilySize': 2},
    {'name': 'Niño, 3ra Clase', 'Pclass': 3, 'Sex': 'male', 'Age': 8, 'FamilySize': 5},
    {'name': 'Adulto Mayor, 2da Clase', 'Pclass': 2, 'Sex': 'male', 'Age': 65, 'FamilySize': 1}
]

scenario_probs = []
for scenario in scenarios:
    prob = predict_survival_probability(pd.Series(scenario))
    scenario_probs.append({'Escenario': scenario['name'], 'Probabilidad': prob * 100})

scenario_df = pd.DataFrame(scenario_probs)

fig = px.bar(scenario_df, 
             x='Probabilidad', 
             y='Escenario', 
             orientation='h',
             title="Probabilidades de Supervivencia por Escenario",
             color='Probabilidad',
             color_continuous_scale='RdYlGn')
fig.update_layout(height=500)
fig.show()

# Análisis Prescriptivo

## Row

In [ ]:
#| title: "Recomendaciones Estratégicas para Situaciones de Emergencia"

recommendations = pd.DataFrame({
    'Prioridad': ['Alta', 'Alta', 'Media', 'Media', 'Baja'],
    'Acción': [
        'Evacuar mujeres y niños primero',
        'Garantizar acceso prioritario a pasajeros de primera clase',
        'Implementar protocolos especiales para familias grandes',
        'Capacitación específica para adultos mayores',
        'Revisión de políticas de evacuación por género'
    ],
    'Impacto_Esperado': [95, 85, 70, 60, 40],
    'Costo_Implementacion': ['Bajo', 'Medio', 'Alto', 'Medio', 'Bajo']
})

fig = px.scatter(recommendations, 
                x='Impacto_Esperado', 
                y='Prioridad',
                size=[100, 90, 70, 60, 40],
                color='Costo_Implementacion',
                hover_data=['Acción'],
                title="Matriz de Recomendaciones: Impacto vs Prioridad")
fig.update_layout(height=400)
fig.show()

::: {.card}
### Conclusiones y Recomendaciones

**Hallazgos principales:**
- La clase socioeconómica fue el factor más determinante en la supervivencia
- El género mostró una clara diferencia en las tasas de supervivencia
- Los grupos familiares medianos tuvieron mejor tasa de supervivencia

**Recomendaciones operativas:**
1. **Protocolos de evacuación** basados en análisis de riesgo
2. **Capacitación diferenciada** por grupos demográficos
3. **Sistemas de alerta temprana** para familias grandes
4. **Revisión de políticas** de emergencia marítima

**Próximos pasos:**
- Implementar modelo predictivo en tiempo real
- Desarrollar dashboard operativo para emergencias
- Validar recomendaciones con datos históricos adicionales
:::

## Row {height=30%}

::: {.card}
### Metodología Aplicada

Este análisis sigue el framework de **4 tipos de analítica** aprendido en clase:
- **Descriptivo**: ¿Qué pasó?
- **Diagnóstico**: ¿Por qué pasó?
- **Predictivo**: ¿Qué pasará?
- **Prescriptivo**: ¿Qué debemos hacer?

Utilizando datos del Titanic como caso de estudio para demostrar técnicas de análisis de datos y toma de decisiones conforme a los contenidos del curso **"Análisis de la información y la Decisión"**.
:::

# Reflexión Académica

## Row

::: {.card}
### Conexión con los Contenidos del Curso

Basándose en las notas de clase proporcionadas, este dashboard implementa varios conceptos clave:

**🧠 Modelos y Apoyo para la Toma de Decisiones:**
- Se aplicaron los 4 tipos de análisis (descriptivo, diagnóstico, predictivo, prescriptivo)
- Se utilizó el enfoque de **recolección de datos** y **evaluación de alternativas**
- Se implementó la separación entre **inteligencia operativa** (dashboard interactivo) e **inteligencia empresarial** (análisis histórico)

**📊 Análisis de Datos y Aplicaciones Prácticas:**
- Se siguió el modelo del profesor **Gino Taipe Miranda** con ejercicios prácticos del dataset Titanic
- Se aplicó la premisa de que el **análisis prescriptivo depende del predictivo**
- Se utilizaron herramientas modernas como **Python y Plotly** para visualización

**🧾 Reportes vs Dashboards:**
- Se optó por un **dashboard visual** en lugar de un reporte extenso
- Se priorizó la **fácil interpretación** sobre el detalle exhaustivo
- Se balanceó la **claridad con la profundidad analítica**
:::

::: {.card}
### Elementos para la Toma de Decisiones Implementados

**⚙️ Los 4 Elementos Esenciales:**
1. **Humanos**: Dashboard diseñado para diferentes tipos de usuarios
2. **Tareas**: Análisis estructurado desde descriptivo hasta prescriptivo
3. **Datos**: Dataset del Titanic como materia prima
4. **Tecnología**: Quarto + Python + Plotly adaptado a dispositivos web

**🎨 Experiencia de Usuario (UX):**
- **Accesibilidad**: Navegación clara y estructura lógica
- **Claridad**: Visualizaciones interpretables
- **Credibilidad**: Datos verificables y metodología transparente
- **Consistencia**: Estilo visual uniforme
- **Usabilidad**: Interfaz intuitiva con elementos interactivos
:::

::: {.card}
### Data Storytelling Aplicado

**📚 Método Implementado:**
- **Estructura Minto**: Idea principal (supervivencia del Titanic) → Razones (factores de riesgo) → Evidencias (análisis estadístico)
- **Narrativa progresiva**: Desde "¿qué pasó?" hasta "¿qué debemos hacer?"
- **Impacto emocional**: Uso de colores, iconos y visualizaciones que conectan con la audiencia

**🛠️ Tecnologías Utilizadas:**
- **Quarto Dashboard**: Para estructura y presentación
- **Python**: Para análisis de datos
- **Plotly**: Para visualizaciones interactivas
- **CSS personalizado**: Para mantener identidad institucional

Esta implementación demuestra la aplicación práctica de todos los conceptos del curso en un proyecto real de análisis de datos.
:::